# Wiki Scrape

a script to scrape star names off of wikipedia's list of stars, therefore assuring that data on pageviews can be found

In [1]:
## imports

import requests
from bs4 import BeautifulSoup
import random
import pandas as pd
from io import StringIO

# Data Extraction

In [2]:
## constants

HEADERS = {'User-Agent': 'CelestialCelebs (sw2499@cornell.edu)'}

In [ ]:
## a function to get data from wikipedia api
#
# params:
#   PAGE: the name of the Wikipedia page to fetch data for
#   FILE_PATH: the path to the file where the fetched data will be saved
# returns:
#   None

def get_wiki_data(page, file_path):
    r = requests.get(
        'https://en.wikipedia.org/w/api.php',
        params={'action': 'parse', 'format': 'json', 'page': page, 'prop': 'text', 'formatversion': 2},
        headers=HEADERS,
        timeout=60
    )
    r.raise_for_status()
    data = r.json()['parse']['text']

    # write to a file so we can inspect data without repeatedly scraping
    f = open(file_path, 'w')
    f.write(data)
    f.close()

In [ ]:
get_wiki_data('List_of_proper_names_of_stars', 'stars.html')
get_wiki_data('List_of_proper_names_of_exoplanets', 'exoplanets.html')
get_wiki_data('List_of_galaxies', 'galaxies.html')

# Data Parsing

## Stars

main useful info: star name (if there is one) and official designation

In [ ]:
## read data

star_data = open('stars.html', 'r').read()

## random seed

RANDOM_SEED = 8
random.seed(RANDOM_SEED)

In [ ]:
## parse html data and fix star names and designations

stars_df = pd.read_html(StringIO(star_data), match='Modern proper name', extract_links='body')[0] # read first table matching "proper name"
N_STARS = stars_df.shape[0] # total number of rows
stars_df['Linked'] = False # by default assume there's no link

for r in range(1, N_STARS):
    row = stars_df.loc[r]
    
    if ((row['Modern proper name'][1] is not None) and
        (row['Modern proper name'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        stars_df.loc[r, 'Linked'] = True
    stars_df.loc[r, 'Modern proper name'] = row['Modern proper name'][0] # fix name
    
    if ((row['Designation'][1] is not None) and
        (row['Designation'][1][:6] == '/wiki/')):
        # if the designation is linked, the entry is linked
        stars_df.loc[r, 'Linked'] = True
    stars_df.loc[r, 'Designation'] = row['Designation'][0] # fix designation

In [ ]:
## a function to randomly select stars from the table, given that the star has a valid hyperlink to another page
#
# params:
#   n: number of random stars to select
# returns:
#   2D list containing [common_name, designation] for each selected star

def select_random_stars(n):
    objs = []
    row_idxs = random.sample(range(1, N_STARS), n)
    for idx in row_idxs:
        # get row from table
        row = stars_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Modern proper name'], row['Designation']])
        
    return pd.DataFrame(objs, columns=['common_name', 'designation'])

In [ ]:
## select 200 stars and save as a csv
# select_random_stars(200).to_csv('stars.csv', index=False)
# file exists now, don't need to run it again

## Exoplanets

main useful info: planet name and host star name

In [57]:
## read data

exoplanet_data = open('exoplanets.html', 'r').read()

In [64]:
## parse html data and fix exoplanet names and designations

planets_df = pd.read_html(StringIO(exoplanet_data), match='Name of planet', extract_links='body')[0] # read first table matching "name of planet"
N_PLANETS = planets_df.shape[0] # total number of rows
planets_df['Linked'] = False # by default assume there's no link

for r in range(1, N_PLANETS):
    row = planets_df.loc[r]
    
    if ((row['Name of planet'][1] is not None) and
        (row['Name of planet'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        planets_df.loc[r, 'Linked'] = True
    planets_df.loc[r, 'Name of planet'] = row['Name of planet'][0] # fix name

    # if star is linked, that doesn't mean the planet is linked, but we can fix the formatting
    planets_df.loc[r, 'Name of star'] = row['Name of star'][0] # fix name of star

In [67]:
## a function to randomly select exoplanets from the table, given that the exoplanet has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [common name, host star] for each selected exoplanet

def select_random_planets(n):
    objs = []
    row_idxs = random.sample(range(1, N_PLANETS), n)
    for idx in row_idxs:
        # get row from table
        row = planets_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Name of planet'], row['Name of star']])
        
    return pd.DataFrame(objs, columns=['common_name', 'host_star'])

In [ ]:
## select 75 exoplanets and save as a csv
select_random_planets(75).to_csv('exoplanets.csv', index=False)

## Galaxies

main useful info: name

In [71]:
## read data

galaxy_data = open('galaxies.html', 'r').read()

In [72]:
## parse html data and fix exoplanet names and designations

galaxies_df = pd.read_html(StringIO(galaxy_data), match='Galaxy', extract_links='body')[0] # read first table matching "galaxy"
N_GALAXIES = galaxies_df.shape[0] # total number of rows
galaxies_df['Linked'] = False # by default assume there's no link

for r in range(1, N_GALAXIES):
    row = galaxies_df.loc[r]
    
    if ((row['Galaxy'][1] is not None) and
        (row['Galaxy'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        galaxies_df.loc[r, 'Linked'] = True
    galaxies_df.loc[r, 'Galaxy'] = row['Galaxy'][0] # fix name

In [73]:
## a function to randomly select galaxies from the table, given that the galaxy has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [official name] for each selected exoplanet

def select_random_galaxies(n):
    objs = []
    row_idxs = random.sample(range(1, N_GALAXIES), n)
    for idx in row_idxs:
        # get row from table
        row = galaxies_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Galaxy']])
        
    return pd.DataFrame(objs, columns=['official_name'])

In [74]:
## select 75 galaxies and save as a csv
select_random_galaxies(50).to_csv('galaxies.csv', index=False)